# Spectral Matching - Closing the Real/Fake Gap

**Paper role: Section 4 (attack / robustness).**

Notebooks 1-2 established the finding: diffusion-generated images preserve
low-frequency structure but carry a **high-frequency energy deficit** (the
~21x high-band gap; log-log slope steeper than real's ~1/f).

Here we test whether that gap is a *robust* detector or a *cheap-to-remove*
artifact. We measure the average radial magnitude profile of **real** images,
then rewrite a fake's Fourier **magnitude** along each radius to match it,
leaving the **phase untouched** (phase = structure, so content is preserved).

Result to report: the high-band gap collapses toward 1x and the slope moves to
the real range, i.e. a frequency-only detector is defeated by a single FFT
post-process. This motivates multi-cue detection (phase, learned, pixel-stat),
which this attack does **not** defeat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- inline copy of src/spectral.py so the notebook runs standalone (e.g. Colab) ---
LOW_MID_EDGE, MID_HIGH_EDGE = 20, 60

def load_gray(path, size=256):
    return np.asarray(Image.open(path).convert('L').resize((size, size)), dtype=np.float64)

def load_rgb(path, size=256):
    return np.asarray(Image.open(path).convert('RGB').resize((size, size)), dtype=np.float64)

def _radius_map(h, w):
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    return np.round(np.sqrt((y - cy) ** 2 + (x - cx) ** 2)).astype(int)

def radial_profile(channel):
    F = np.fft.fftshift(np.fft.fft2(channel))
    mag = np.abs(F)
    r = _radius_map(*channel.shape)
    max_r = min(channel.shape) // 2
    total = np.bincount(r.ravel(), weights=mag.ravel())
    count = np.bincount(r.ravel())
    return total[:max_r] / np.maximum(count[:max_r], 1)

def radial_target(paths, size=256, color=False):
    profs = []
    for p in paths:
        if color:
            arr = load_rgb(p, size)
            profs.append(np.stack([radial_profile(arr[..., c]) for c in range(3)]))
        else:
            profs.append(radial_profile(load_gray(p, size)))
    return np.mean(profs, axis=0)

def _match_channel(channel, target, gain_clip=(0.1, 12.0), smooth=3, preserve_dc=True):
    F = np.fft.fftshift(np.fft.fft2(channel))
    r = _radius_map(*channel.shape)
    max_r = len(target)
    gain = target / (radial_profile(channel) + 1e-12)
    if smooth and smooth > 1:
        gain = np.convolve(gain, np.ones(smooth) / smooth, mode='same')
    gain = np.clip(gain, *gain_clip)
    if preserve_dc:
        gain[0] = 1.0
    gmap = gain[np.clip(r, 0, max_r - 1)]
    gmap[r >= max_r] = 1.0
    if preserve_dc:
        gmap[r == 0] = 1.0
    out = np.fft.ifft2(np.fft.ifftshift(F * gmap)).real
    return out

def spectral_match(img, target, gain_clip=(0.1, 12.0), smooth=3, clip_out=True, preserve_dc=True):
    if img.ndim == 2:
        out = _match_channel(img, target, gain_clip, smooth, preserve_dc)
    else:
        out = np.stack([_match_channel(img[..., c], target[c], gain_clip, smooth, preserve_dc)
                        for c in range(img.shape[-1])], axis=-1)
    return np.clip(out, 0, 255) if clip_out else out

def spectral_report(channel):
    prof = radial_profile(channel)
    freqs = np.arange(1, len(prof))
    slope, _ = np.polyfit(np.log(freqs + 1e-8), np.log(prof[1:] + 1e-8), 1)
    return {'low': float(prof[:LOW_MID_EDGE].mean()),
            'mid': float(prof[LOW_MID_EDGE:MID_HIGH_EDGE].mean()),
            'high': float(prof[MID_HIGH_EDGE:].mean()),
            'slope': float(slope), 'profile': prof}

print('helpers ready')

## Config

Point these at the files Notebook 1 saved. `REAL_PATHS` are the CIFAR reals;
`FAKE_PATH` is the Stable Diffusion output you want to make 'pass'.

In [ ]:
SIZE       = 256
REAL_PATHS = [f'/content/real_{i}.png' for i in range(5)]
FAKE_PATH  = '/content/fake_sd15.png'   # <-- set to your SD fake from Notebook 1
OUT_GRAY   = '/content/fake_matched_gray.png'
OUT_COLOR  = '/content/fake_matched_color.png'

import os
missing = [p for p in REAL_PATHS + [FAKE_PATH] if not os.path.exists(p)]
assert not missing, f'Missing files (run Notebook 1 first / fix paths): {missing}'
print('inputs found')

## Match and report (luminance)

Analysis is done on luminance to match Notebook 1's metrics.

In [ ]:
target = radial_target(REAL_PATHS, size=SIZE, color=False)

real_ref = load_gray(REAL_PATHS[0], SIZE)
fake     = load_gray(FAKE_PATH, SIZE)
matched  = spectral_match(fake, target)

rep_real, rep_fake, rep_match = map(spectral_report, (real_ref, fake, matched))

print(f"{'metric':<8}{'real':>14}{'fake':>14}{'matched':>14}")
for k in ('low', 'mid', 'high', 'slope'):
    print(f'{k:<8}{rep_real[k]:>14.3f}{rep_fake[k]:>14.3f}{rep_match[k]:>14.3f}')

gap_before = rep_real['high'] / rep_fake['high']
gap_after  = rep_real['high'] / rep_match['high']
print(f'\nHigh-band gap real/fake    : {gap_before:6.2f}x')
print(f'High-band gap real/matched : {gap_after:6.2f}x   <- closed by spectral match')

Image.fromarray(matched.astype(np.uint8)).save(OUT_GRAY)
print('saved', OUT_GRAY)

## Headline figure (Figure 4)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Spectral matching closes the high-frequency gap (phase preserved)',
             fontsize=14, fontweight='bold')

for a, img, t in zip(ax[0], (real_ref, fake, matched), ('Real', 'Fake (SD)', 'Fake -> matched')):
    a.imshow(img, cmap='gray', vmin=0, vmax=255); a.set_title(t); a.axis('off')

# radial profiles: before vs after
for a, prof_fake_or_match, lbl in (
        (ax[1, 0], rep_fake,  'Fake'),
        (ax[1, 1], rep_match, 'Matched')):
    a.semilogy(rep_real['profile'], color='blue', lw=2.5, label='Real')
    a.semilogy(prof_fake_or_match['profile'], color='red', lw=2.5, ls='--', label=lbl)
    a.axvspan(0, 20, alpha=0.08, color='green')
    a.axvspan(20, 60, alpha=0.08, color='yellow')
    a.axvspan(60, 128, alpha=0.08, color='orange')
    a.set_title(f'Radial profile: Real vs {lbl}')
    a.set_xlabel('radius (frequency)'); a.set_ylabel('mean |FFT|'); a.legend()

# band bar chart
bands = ['Low', 'Mid', 'High']
x = np.arange(3); w = 0.25
ax[1, 2].bar(x - w, [rep_real[k]  for k in ('low','mid','high')], w, label='Real',    color='blue')
ax[1, 2].bar(x,     [rep_fake[k]  for k in ('low','mid','high')], w, label='Fake',    color='red')
ax[1, 2].bar(x + w, [rep_match[k] for k in ('low','mid','high')], w, label='Matched', color='green')
ax[1, 2].set_yscale('log'); ax[1, 2].set_xticks(x); ax[1, 2].set_xticklabels(bands)
ax[1, 2].set_title('Band energy: Real / Fake / Matched'); ax[1, 2].legend()

plt.tight_layout()
plt.savefig('/content/fig4_spectral_matching.png', dpi=150, bbox_inches='tight')
plt.show()

## Color 'perfect fake' (per-channel)

Match each RGB channel to its own real target and save a viewable color result.

In [ ]:
target_rgb = radial_target(REAL_PATHS, size=SIZE, color=True)   # (3, R)
fake_rgb   = load_rgb(FAKE_PATH, SIZE)
matched_rgb = spectral_match(fake_rgb, target_rgb)

Image.fromarray(matched_rgb.astype(np.uint8)).save(OUT_COLOR)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, img, t in zip(ax, (load_rgb(REAL_PATHS[0], SIZE), fake_rgb, matched_rgb),
                     ('Real', 'Fake (SD)', 'Fake -> matched (color)')):
    a.imshow(img.astype(np.uint8)); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()
print('saved', OUT_COLOR)

## Interpretation (draft text for Section 4)

- A single radial-magnitude rewrite drives the high-band gap from ~21x to ~1x
  and the log-log slope into the real range, **without touching phase** - so the
  visible content is unchanged while the spectral fingerprint is removed.
- **Therefore the radial power spectrum alone is not a robust detector.** Any
  classifier keyed on band energy or slope can be evaded by this O(1 FFT)
  post-process.
- **Caveat / scope (state this explicitly in the paper):** matching the
  *azimuthally-averaged* magnitude does not equalize the full 2-D spectrum,
  leaves **phase statistics** untouched, and does not address learned or
  pixel-domain detectors. Report those as the directions that survive the attack.